# Entertainment Mode vs Ranked Mode — Riot API Data Collector
**Design:** Case-Control | Cases = played Entertainment Mode | Controls = did not  
**Sampling:** Stratified by ranked tier, balanced 8 cases + 8 controls per tier

## 0. Install dependencies

In [1]:
!pip install requests pandas tqdm -q

## 1. Configuration — fill in your API key here

In [81]:
API_KEY  = "RGAPI-9755c32e-c002-4ab3-9261-578a1e031393"   # ← paste your Riot API key
REGION   = "na1"                    # server: na1 / euw1 / kr / eun1 / br1 / jp1 / la1 / la2
ROUTE    = "americas"               # routing: americas (na1/br1/la1/la2) | europe (euw1/eun1/tr1/ru) | asia (kr/jp1)



TIERS = ["IRON", "BRONZE", "SILVER", "GOLD", "PLATINUM", "DIAMOND"]
# Add "EMERALD" between PLATINUM and DIAMOND if you want it (added in 2023 season)

## 2. Auto-detect Hextech ARAM queue ID from Riot's official queue list

In [3]:
import requests
import pandas as pd

QUEUES_URL = "https://static.developer.riotgames.com/docs/lol/queues.json"

resp = requests.get(QUEUES_URL)
queues = resp.json()

# Show all queues containing "ARAM" so we can confirm IDs
aram_queues = [q for q in queues if "ARAM" in (q.get("description") or "").upper()
                                 or "ARAM" in (q.get("notes") or "").upper()]

df_aram = pd.DataFrame(aram_queues)[["queueId", "map", "description", "notes"]]
print("All ARAM-related queues from Riot's official list:")
df_aram

All ARAM-related queues from Riot's official list:


,queueId,map,description,notes
0,65,Howling Abyss,5v5 ARAM games,Deprecated in patch 7.19 in favor of queueId 450
1,67,Howling Abyss,ARAM Co-op vs AI games,Game mode deprecated
2,100,Butcher's Bridge,5v5 ARAM games,None
3,450,Howling Abyss,5v5 ARAM games,None
4,720,Howling Abyss,ARAM Clash games,None
5,2400,Howling Abyss,ARAM: Mayhem,None


In [19]:
df_queues = pd.DataFrame(queues)
pd.set_option('display.max_rows', None)
df_queues_filtered_deprecated = df_queues[~df_queues['notes'].str.contains('deprecated', case=False, na=False)]
df_queues_filtered_deprecated

,queueId,map,description,notes
0,0,Custom games,None,None
21,72,Howling Abyss,1v1 Snowdown Showdown games,None
22,73,Howling Abyss,2v2 Snowdown Showdown games,None
23,75,Summoner's Rift,6v6 Hexakill games,None
24,76,Summoner's Rift,Ultra Rapid Fire games,None
25,78,Howling Abyss,One For All: Mirror Mode games,None
26,83,Summoner's Rift,Co-op vs AI Ultra Rapid Fire games,None
31,98,Twisted Treeline,6v6 Hexakill games,None
32,100,Butcher's Bridge,5v5 ARAM games,None
34,310,Summoner's Rift,Nemesis games,None



Based on Riot's official `queues.json` (verified March 2025).

| Category | Queue IDs | Notes |
|---|---|---|
| **Hextech ARAM** | `2400` | "ARAM: Mayhem" — our case indicator |
| **Ranked** | `420`, `440`, `700`, `720` | Solo/Duo, Flex, Clash, ARAM Clash |
| **Normal Match** | `400`, `430`, `480`, `490` | Draft, Blind, Swiftplay, Quickplay |
| **ARAM (standard)** | `450`, `100` | Howling Abyss + Butcher's Bridge |
| **Special / rotating** | `900`, `1020`, `1300`, `1400`, `1700`, `1710`, `1900`, `2300` | URF, One for All, Nexus Blitz, Arena, etc. |
| **Bot / co-op** | `820`, `870`, `880`, `890` | Excluded from analysis |
| **TFT / Tutorial / Swarm** | `1090–1210`, `1810–1840`, `2000–2020` | Excluded entirely |

In [46]:
# ── Queue ID sets ────────────────────────────────────────────────────────────
RANKED = {420, 440}
MATCH = {400, 430, 480, 490}
CASUAL = {
    450, 100,          # ARAM
    900, 1010, 1900,   # URF
    1020,              # One for All
    1300,              # Nexus Blitz
    1400,              # Spellbook
    1700, 1710,        # Arena
    2300               # Brawl
}
BOTS = {820, 870, 880, 890}

def categorize_queue(qid):
    if qid in {420, 440}:
        return "ranked"
    elif qid in {400, 430, 480, 490}:
        return "match"
    elif qid in {450, 100, 900, 1010, 1900, 1020, 1300, 1400, 1700, 1710, 2300}:
        return "casual_pvp"
    elif qid in {820, 870, 880, 890}:
        return "bot"
    else:
        return "exclude"

## 3. API helpers — rate-limit safe GET

In [21]:
import time
import random

HEADERS = {"X-Riot-Token": API_KEY}

def get(url: str, params: dict = None, retries: int = 6):
    """GET with automatic 429 retry + exponential backoff."""
    for attempt in range(retries):
        resp = requests.get(url, headers=HEADERS, params=params or {})
        if resp.status_code == 200:
            return resp.json()
        elif resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", 2 ** (attempt + 1)))
            print(f"  [429] Rate limited → waiting {wait}s …")
            time.sleep(wait)
        elif resp.status_code == 404:
            return None
        else:
            print(f"  [HTTP {resp.status_code}] {url}")
            time.sleep(1.5)
    return None

def pause(s: float = 1.2):
    """Stay within Riot's default 20 req/s dev key limit."""
    time.sleep(s)

print("API helpers ready.")

API helpers ready.


## 4. Step 1 — Fetch player PUUIDs by tier (League-V4)

In [86]:
import random

def fetch_puuids_for_tier(tier: str, n_per_division: int, max_try_pages: int = 20) -> list:
    print(f"  Fetching {tier} …")

    puuids = []
    seen = set()

    for div in ["I", "II", "III", "IV"]:
        count = 0
        tried_pages = set()

        while count < n_per_division and len(tried_pages) < max_try_pages:
            # 👉 随机选一个 page（核心）
            page = random.randint(1, max_try_pages)
            if page in tried_pages:
                continue
            tried_pages.add(page)

            url = f"https://{REGION}.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/{tier}/{div}"
            entries = get(url, params={"page": page})
            pause()

            if not entries:
                continue

            random.shuffle(entries)

            for entry in entries:
                if count >= n_per_division:
                    break

                puuid = entry.get("puuid")
                if puuid and puuid not in seen:
                    puuids.append(puuid)
                    seen.add(puuid)
                    count += 1

        print(f"    {tier} {div}: {count} players")

    print(f"  ✓ {len(puuids)} PUUIDs collected for {tier}")
    return puuids

## 5. Step 2 — Fetch match IDs (Match-V5)

In [49]:
def fetch_match_ids(puuid: str, count: int = MATCHES_PER_PLAYER) -> list:
    """Return the last `count` match IDs for a player (all queues)."""
    url = f"https://{ROUTE}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids"
    data = get(url, params={"count": count})
    pause(0.6)
    return data if data else []

print("fetch_match_ids() defined.")

fetch_match_ids() defined.


## 6. Step 3 — Fetch match details (Match-V5)

In [50]:
def fetch_match(match_id: str) -> dict:
    """Fetch full match detail from Match-V5."""
    url = f"https://{ROUTE}.api.riotgames.com/lol/match/v5/matches/{match_id}"
    data = get(url)
    pause(0.6)
    return data

print("fetch_match() defined.")

fetch_match() defined.


## Step 4 — Filter match after Oct 22, 2025

### MVP看需要sample多少才够match

In [78]:
def fetch_sample_puuids(tier="GOLD", division="I", n=5):
    url = f"https://na1.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/{tier}/{division}"
    
    res = requests.get(url, headers=HEADERS)
    
    if res.status_code != 200:
        print("❌ error:", res.status_code)
        return []
    
    data = res.json()
    
    puuids = []
    for entry in data[:n]:
        if "puuid" in entry:
            puuids.append(entry["puuid"])
    
    print(f"✅ got {len(puuids)} puuids")
    return puuids

In [77]:
def fetch_multi_tier_puuids():
    tiers = ["SILVER", "GOLD", "PLATINUM"]
    puuids = []

    for t in tiers:
        puuids += fetch_sample_puuids(tier=t, n=3)

    return puuids

In [79]:
import requests
import time
import datetime

CUTOFF = int(datetime.datetime(2025, 10, 22).timestamp() * 1000)


def get_match_ids(puuid, count=100):
    url = f"https://{ROUTE}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids"
    params = {"start": 0, "count": count}
    return requests.get(url, headers=HEADERS, params=params).json()


def get_match_end_time(match_id):
    url = f"https://{ROUTE}.api.riotgames.com/lol/match/v5/matches/{match_id}"
    res = requests.get(url, headers=HEADERS)

    if res.status_code != 200:
        return None
    
    data = res.json()
    return data["info"].get("gameEndTimestamp")


def test_time_filter(puuid_list, count_per_user=100):
    total = 0
    valid = 0

    for puuid in puuid_list:
        match_ids = get_match_ids(puuid, count_per_user)

        if not match_ids:
            continue

        for mid in match_ids:
            t = get_match_end_time(mid)
            if t is None:
                continue

            total += 1

            if t < CUTOFF:
                valid += 1

            time.sleep(0.1)  # 防429

    print("\n====== RESULT ======")
    print("Total matches:   ", total)
    print("Before Oct 22:   ", valid)
    print("Filtered out:    ", total - valid)
    print("Keep ratio:      ", round(valid / total, 3) if total else 0)

In [80]:
puuids = fetch_multi_tier_puuids()
test_time_filter(puuids)

✅ got 3 puuids
✅ got 3 puuids
✅ got 3 puuids

====== RESULT ======
Total matches:    99
Before Oct 22:    69
Filtered out:     30
Keep ratio:       0.697


In [82]:
def get_match_data(match_id):
    """拉完整 match 数据，返回 (timestamp, data_dict)"""
    url = f"https://{ROUTE}.api.riotgames.com/lol/match/v5/matches/{match_id}"
    res = requests.get(url, headers=HEADERS)
    if res.status_code != 200:
        return None, None

    data = res.json()
    t = data["info"].get("gameEndTimestamp")
    return t, data


def process_matches_for_player(puuid, match_ids):
    rows_total = []
    rows_valid = []
    total = 0

    for mid in match_ids:
        t, data = get_match_data(mid)
        if t is None:
            continue

        total += 1

        # 你想保存的字段，按需增删
        row = {
            "match_id":        mid,
            "puuid":           puuid,
            "gameEndTimestamp": t,
            "gameDuration":    data["info"].get("gameDuration"),
            "gameMode":        data["info"].get("gameMode"),
            "gameVersion":     data["info"].get("gameVersion"),
            # 如果需要参与者数据可以展开 participants
            # "participants": data["info"].get("participants"),
        }

        rows_total.append(row)
        if t < CUTOFF:
            rows_valid.append(row)

        time.sleep(0.1)

    return {
        "puuid":       puuid,
        "total":       total,
        "valid":       len(rows_valid),
        "filtered":    total - len(rows_valid),
        "rows_total":  rows_total,
        "rows_valid":  rows_valid,
    }

## 8. Main collection loop

这里拉了哪些variable还要进一步确定

In [83]:
SAMPLES_PER_TIER   = 1000    # cases AND controls each
MATCHES_PER_PLAYER = 100   # last N matches to pull

In [87]:
all_stats  = []
total_rows = []
valid_rows = []

for tier in TIERS:
    print(f"\n{'='*50}")
    print(f"  TIER: {tier}")
    print(f"{'='*50}")

    puuids = fetch_puuids_for_tier(tier, n_per_division=SAMPLES_PER_TIER)

    for puuid in tqdm(puuids, desc=f"{tier} players"):
        match_ids = get_match_ids(puuid, MATCHES_PER_PLAYER)
        if not match_ids:
            continue

        stats = process_matches_for_player(puuid, match_ids)
        all_stats.append(stats)

        total_rows.extend(stats["rows_total"])  # ← 新增
        valid_rows.extend(stats["rows_valid"])  # ← 新增

# 汇总统计
total_all    = sum(r["total"]    for r in all_stats)
valid_all    = sum(r["valid"]    for r in all_stats)
filtered_all = sum(r["filtered"] for r in all_stats)

print("\n====== FINAL RESULT ======")
print("Total matches:   ", total_all)
print("Before Oct 22:   ", valid_all)
print("Filtered out:    ", filtered_all)
print("Keep ratio:      ", round(valid_all / total_all, 3) if total_all else 0)

# 构建 DataFrame 并保存
df_total = pd.DataFrame(total_rows)
df_valid = pd.DataFrame(valid_rows)

print(f"\ndf_total shape: {df_total.shape}")
print(f"df_valid shape: {df_valid.shape}")

df_total.to_parquet("matches_total.parquet", index=False)
df_valid.to_parquet("matches_valid.parquet", index=False)


  TIER: IRON
  Fetching IRON …
    IRON I: 1000 players
    IRON II: 1000 players
    IRON III: 1000 players
    IRON IV: 1000 players
  ✓ 4000 PUUIDs collected for IRON


IRON players:   0%|          | 0/4000 [00:00<?, ?it/s]


  TIER: BRONZE
  Fetching BRONZE …
    BRONZE I: 1000 players
    BRONZE II: 1000 players
    BRONZE III: 1000 players
    BRONZE IV: 1000 players
  ✓ 4000 PUUIDs collected for BRONZE


BRONZE players:   0%|          | 0/4000 [00:00<?, ?it/s]


  TIER: SILVER
  Fetching SILVER …
    SILVER I: 1000 players
    SILVER II: 1000 players
    SILVER III: 1000 players
    SILVER IV: 1000 players
  ✓ 4000 PUUIDs collected for SILVER


...


  TIER: PLATINUM
  Fetching PLATINUM …
    PLATINUM I: 1000 players
    PLATINUM II: 1000 players
    PLATINUM III: 1000 players
    PLATINUM IV: 1000 players
  ✓ 4000 PUUIDs collected for PLATINUM


PLATINUM players:   0%|          | 0/4000 [00:00<?, ?it/s]


  TIER: DIAMOND
  Fetching DIAMOND …
    DIAMOND I: 1000 players
    DIAMOND II: 1000 players
    DIAMOND III: 1000 players
    DIAMOND IV: 1000 players
  ✓ 4000 PUUIDs collected for DIAMOND


DIAMOND players:   0%|          | 0/4000 [00:00<?, ?it/s]


====== FINAL RESULT ======
Total matches:    9542
Before Oct 22:    1045
Filtered out:     8497
Keep ratio:       0.11


## 9. Save to CSV

In [88]:
import pandas as pd
import json
from datetime import datetime

# 👉 转 DataFrame
df_stats = pd.DataFrame(all_stats)

# 👉 时间戳命名（防止覆盖）
ts = datetime.now().strftime("%Y%m%d_%H%M")

# 👉 存 CSV（方便看）
csv_path = f"match_stats_{ts}.csv"
df_stats.to_csv(csv_path, index=False)

# 👉 存 JSON（保留完整结构，特别是 valid_ids）
json_path = f"match_stats_{ts}.json"
with open(json_path, "w") as f:
    json.dump(all_stats, f)

print(f"✅ Saved to:\n- {csv_path}\n- {json_path}")

✅ Saved to:
- match_stats_20260329_1826.csv
- match_stats_20260329_1826.json


In [89]:
df_stats.head()

,puuid,total,valid,filtered,valid_ids
0,M-YskURGdZXfU_qn4pZ3dSpQ8LFY020SHyTnWTok65Btyq...,99,0,99,[]
1,7JuhRZd_ERyOvbkEkqrj4AMAel29x2w2F5W_t-zUgpLi6j...,0,0,0,[]
2,LZEiZn9HGQE6kyCIPVw9C9HIW-7AtjwhZtnkT3Wbp2tT0i...,0,0,0,[]
3,VGocl-jiiyF_Xh1g8RC_celJzfp0RCVNmFIXDIYKNIDo6O...,0,0,0,[]
4,8JNxJRgAEpIsgXMcodQStA4-0i0aWeuWD9U9hXUU09OZeh...,0,0,0,[]
